# 05c — LoRA: SmolLM3-3B + GPT-2 sweeps + transfer

**Project:** Lightweight domain adaptation of small LLMs for chemistry using LoRA and QLoRA (MSc FYP).

**This notebook (05c):** finishes Phase 5 by running the supervisor's full LoRA recipe for the remaining two models:

```
For each model in [SmolLM3-3B, GPT-2]:
  1. BBBP sweep: rank ∈ {8, 16} × lr ∈ {1e-4, 5e-4}    = 4 runs
  2. Pick best config by ROC-AUC                       = 1 winner
  3. Apply winner to BACE + ESOL                       = 2 runs
= 6 runs per model × 2 models                          = 12 LoRA rows
```

**After this notebook the Phase 5 scoreboard contains 18 LoRA rows** (6 from 05b + 12 from here).

**Estimated runtime on T4:** ~60–80 min total. SmolLM3 dominates the cost; GPT-2 is fast.

**Self-contained:** re-implements the harness from 05b so this notebook stands alone.

## 1. Install + imports

In [ ]:
%pip install -q --upgrade torchao
%pip install -q --upgrade transformers peft accelerate bitsandbytes rdkit pandas scikit-learn

In [ ]:
import os, random, gc, time, json
from dataclasses import dataclass
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset as TorchDataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, mean_squared_error, mean_absolute_error
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))
    print("VRAM  :", f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB")

## 2. Mount Drive + paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR     = "/content/drive/MyDrive/chemistry-peft-fyp/data"
RESULTS_DIR  = "/content/drive/MyDrive/chemistry-peft-fyp/results"
ADAPTERS_DIR = "/content/drive/MyDrive/chemistry-peft-fyp/adapters"
RESULTS_CSV  = f"{RESULTS_DIR}/lora_results.csv"
os.makedirs(RESULTS_DIR,  exist_ok=True)
os.makedirs(ADAPTERS_DIR, exist_ok=True)
print("Adapters dir:", ADAPTERS_DIR)
print("Results CSV :", RESULTS_CSV)

## 3. Load splits + dataset configs + helpers

In [ ]:
def load_splits(name):
    p = name.lower()
    return (
        pd.read_csv(f"{DATA_DIR}/{p}_train.csv"),
        pd.read_csv(f"{DATA_DIR}/{p}_val.csv"),
        pd.read_csv(f"{DATA_DIR}/{p}_test.csv"),
    )

splits = {name: load_splits(name) for name in ["BBBP", "BACE", "ESOL"]}
for n, (tr, va, te) in splits.items():
    print(f"{n:5s}: train={len(tr)}  val={len(va)}  test={len(te)}")

ESOL_BUCKETS = [
    ("very_low",  -np.inf, -6.0, -7.5),
    ("low",       -6.0,    -4.0, -5.0),
    ("medium",    -4.0,    -2.0, -3.0),
    ("high",      -2.0,     0.0, -1.0),
    ("very_high",  0.0,    np.inf, 1.0),
]
ESOL_LABEL_WORDS = [b[0] for b in ESOL_BUCKETS]
ESOL_MIDPOINTS   = np.array([b[3] for b in ESOL_BUCKETS], dtype=np.float32)

def esol_continuous_to_bucket(val):
    for label, lo, hi, _ in ESOL_BUCKETS:
        if lo < val <= hi: return label
    return ESOL_BUCKETS[0][0]

@dataclass
class DatasetConfig:
    name: str; task: str; question: str
    label_words: List[str]; positive_word: Optional[str]

DATASETS = {
    "BBBP": DatasetConfig("BBBP", "classification",
        "Does this molecule cross the blood-brain barrier?",
        ["yes", "no"], "yes"),
    "BACE": DatasetConfig("BACE", "classification",
        "Does this molecule inhibit the BACE-1 enzyme?",
        ["yes", "no"], "yes"),
    "ESOL": DatasetConfig("ESOL", "regression",
        "What is the aqueous solubility of this molecule?",
        ESOL_LABEL_WORDS, None),
}

def label_for_prompt(cfg, raw_label):
    if cfg.task == "classification":
        return cfg.positive_word if int(raw_label) == 1 else "no"
    return esol_continuous_to_bucket(float(raw_label))

def build_prompt(cfg, test_smiles):
    return f"{cfg.question}\nSMILES: {test_smiles}\nAnswer:"

IGNORE_INDEX = -100
MAX_LEN = 256

class PromptAnswerDataset(TorchDataset):
    def __init__(self, df, cfg, tokenizer):
        self.cfg = cfg; self.tok = tokenizer
        self.rows = []
        for _, row in df.iterrows():
            prompt = build_prompt(cfg, row["smiles"])
            answer = label_for_prompt(cfg, row["label"])
            self.rows.append((prompt, answer, row["label"]))
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        prompt, answer, _ = self.rows[i]
        prompt_ids = self.tok(prompt, add_special_tokens=False).input_ids
        answer_ids = self.tok(" " + answer, add_special_tokens=False).input_ids
        eos = [self.tok.eos_token_id] if self.tok.eos_token_id is not None else []
        input_ids = (prompt_ids + answer_ids + eos)[:MAX_LEN]
        labels    = ([IGNORE_INDEX] * len(prompt_ids) + answer_ids + eos)[:MAX_LEN]
        attention = [1] * len(input_ids)
        return {
            "input_ids":      torch.tensor(input_ids,  dtype=torch.long),
            "labels":         torch.tensor(labels,     dtype=torch.long),
            "attention_mask": torch.tensor(attention,  dtype=torch.long),
        }

def collate_pad(batch, pad_id):
    max_len = max(len(b["input_ids"]) for b in batch)
    def pad_to(t, val):
        return torch.cat([t, torch.full((max_len - len(t),), val, dtype=t.dtype)])
    return {
        "input_ids":      torch.stack([pad_to(b["input_ids"],      pad_id) for b in batch]),
        "labels":         torch.stack([pad_to(b["labels"],         IGNORE_INDEX) for b in batch]),
        "attention_mask": torch.stack([pad_to(b["attention_mask"], 0) for b in batch]),
    }

print("Helpers ready.")

## 4. Per-model LoRA target modules + batch sizes

GPT-2 uses fused `c_attn` (QKV in one linear). SmolLM3 uses standard Llama-style `q_proj`/`v_proj`. Batch sizes differ because GPT-2 (124M) fits much larger batches than SmolLM3-3B.

In [ ]:
MODEL_CONFIGS = {
    "HuggingFaceTB/SmolLM3-3B": {
        "target_modules":         ["q_proj", "v_proj"],
        "batch_size":             2,
        "gradient_checkpointing": True,
        "short_name":             "smolllm3",
    },
    "gpt2": {
        "target_modules":         ["c_attn"],
        "batch_size":             8,
        "gradient_checkpointing": False,
        "short_name":             "gpt2",
    },
}

## 5. The shared LoRA training/eval function

In [ ]:
@torch.no_grad()
def score_labels(model, tokenizer, prompt, label_words):
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
    out = model(**enc)
    next_logits = out.logits[0, -1, :]
    log_probs = torch.log_softmax(next_logits, dim=-1)
    token_ids_per_label = []
    for w in label_words:
        cands = set()
        for variant in (" " + w, w):
            t = tokenizer(variant, add_special_tokens=False).input_ids
            if t: cands.add(t[0])
        token_ids_per_label.append(sorted(cands))
    label_logps = [max(log_probs[i].item() for i in ids) for ids in token_ids_per_label]
    logps = np.array(label_logps, dtype=np.float64)
    logps -= logps.max()
    probs = np.exp(logps); probs /= probs.sum()
    return probs

def eval_model(model, tokenizer, dataset_name):
    cfg = DATASETS[dataset_name]
    _, _, te_df = splits[dataset_name]
    probs_all, y_true = [], []
    model.eval()
    for _, row in te_df.iterrows():
        probs_all.append(score_labels(model, tokenizer, build_prompt(cfg, row["smiles"]), cfg.label_words))
        y_true.append(row["label"])
    probs_all = np.stack(probs_all); y_true = np.array(y_true)
    if cfg.task == "classification":
        pos_idx = cfg.label_words.index(cfg.positive_word)
        p_pos = probs_all[:, pos_idx]
        pred  = (p_pos >= 0.5).astype(int)
        return {"primary_metric": "roc_auc",
                "primary_value":  float(roc_auc_score(y_true, p_pos)),
                "secondary_metric": "f1",
                "secondary_value":  float(f1_score(y_true, pred))}
    preds_cont = (probs_all * ESOL_MIDPOINTS[None, :]).sum(axis=1)
    y_cont = y_true.astype(np.float32)
    return {"primary_metric": "rmse",
            "primary_value":  float(np.sqrt(mean_squared_error(y_cont, preds_cont))),
            "secondary_metric": "mae",
            "secondary_value":  float(mean_absolute_error(y_cont, preds_cont))}

def lora_train_eval(
    model_id, dataset_name,
    rank, lora_alpha=16, lora_dropout=0.05,
    lr=1e-4, epochs=3,
    save_adapter_to=None,
):
    mc = MODEL_CONFIGS[model_id]
    cfg = DATASETS[dataset_name]
    tr_df, _, _ = splits[dataset_name]

    print(f"\nLoading {model_id} (fp16)...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="auto",
    )
    if mc["gradient_checkpointing"]:
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()

    lora_cfg = LoraConfig(
        r=rank, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
        target_modules=mc["target_modules"], bias="none", task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_cfg)

    trainable, total = 0, 0
    for p in model.parameters():
        total += p.numel()
        if p.requires_grad: trainable += p.numel()
    print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.4f}%)")

    train_ds = PromptAnswerDataset(tr_df, cfg, tokenizer)
    train_loader = DataLoader(
        train_ds, batch_size=mc["batch_size"], shuffle=True,
        collate_fn=lambda b: collate_pad(b, tokenizer.pad_token_id),
    )

    optim = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    total_steps = len(train_loader) * epochs
    sched = get_linear_schedule_with_warmup(
        optim, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps,
    )

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(model.device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            loss.backward()
            optim.step(); sched.step(); optim.zero_grad()
            epoch_loss += loss.item()
            if (step + 1) % 100 == 0:
                print(f"  epoch {epoch+1} step {step+1}/{len(train_loader)}  loss={loss.item():.4f}  ({time.time()-t0:.0f}s)")
        print(f"  epoch {epoch+1}: mean_loss={epoch_loss/len(train_loader):.4f}")
    train_time = time.time() - t0
    peak_mem_mb = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE == "cuda" else 0

    print("Evaluating on test set...")
    metrics = eval_model(model, tokenizer, dataset_name)

    if save_adapter_to:
        os.makedirs(save_adapter_to, exist_ok=True)
        model.save_pretrained(save_adapter_to)
        tokenizer.save_pretrained(save_adapter_to)
        print(f"Adapter saved → {save_adapter_to}")

    row = {
        "model": model_id, "dataset": dataset_name, "task": cfg.task,
        "lora_rank": rank, "lora_alpha": lora_alpha, "lora_lr": lr, "epochs": epochs,
        "trainable_params": trainable, "total_params": total,
        "trainable_pct":   round(100*trainable/total, 4),
        "peak_mem_mb":     round(peak_mem_mb, 1),
        "train_time_s":    round(train_time, 1),
        "metric_primary":  metrics["primary_metric"],
        "value_primary":   round(metrics["primary_value"],   4),
        "metric_secondary": metrics["secondary_metric"],
        "value_secondary":  round(metrics["secondary_value"], 4),
    }
    del model, optim, sched, train_loader, train_ds
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return row

print("LoRA harness ready.")

## 6. Scoreboard append helper

In [ ]:
LORA_COLS = [
    "model", "dataset", "task",
    "lora_rank", "lora_alpha", "lora_lr", "epochs",
    "trainable_params", "total_params", "trainable_pct",
    "peak_mem_mb", "train_time_s",
    "metric_primary", "value_primary",
    "metric_secondary", "value_secondary",
]

def append_lora_row(row):
    new_df = pd.DataFrame([row])
    if os.path.exists(RESULTS_CSV):
        existing = pd.read_csv(RESULTS_CSV)
        key = ["model", "dataset", "lora_rank", "lora_lr"]
        existing = existing[~existing[key].apply(tuple, axis=1).isin(
            new_df[key].apply(tuple, axis=1)
        )]
        combined = pd.concat([existing, new_df], ignore_index=True)
    else:
        combined = new_df
    combined = combined[LORA_COLS]
    combined.to_csv(RESULTS_CSV, index=False)
    print(f"LoRA scoreboard now: {len(combined)} rows.")

## 7. The shared sweep+transfer driver

One function that runs the full Phase 5 recipe for a given model. Reused for SmolLM3 below, then GPT-2.

In [ ]:
SWEEP_GRID = [
    {"rank": 8,  "lr": 1e-4},
    {"rank": 8,  "lr": 5e-4},
    {"rank": 16, "lr": 1e-4},
    {"rank": 16, "lr": 5e-4},
]

def run_full_recipe(model_id):
    short = MODEL_CONFIGS[model_id]["short_name"]
    bbbp_rows = []
    for i, hp in enumerate(SWEEP_GRID, 1):
        adapter_dir = f"{ADAPTERS_DIR}/{short}_bbbp_r{hp['rank']}_lr{hp['lr']:.0e}"
        print(f"\n{'#'*64}\n# {short.upper()} sweep {i}/4 — rank={hp['rank']} lr={hp['lr']:.0e}\n{'#'*64}")
        row = lora_train_eval(
            model_id=model_id, dataset_name="BBBP",
            rank=hp["rank"], lr=hp["lr"], epochs=3,
            save_adapter_to=adapter_dir,
        )
        bbbp_rows.append(row)
        append_lora_row(row)

    sweep_df = pd.DataFrame(bbbp_rows)
    print(f"\n--- {short} BBBP sweep summary ---")
    print(sweep_df[["lora_rank","lora_lr","value_primary","value_secondary","peak_mem_mb","train_time_s"]].to_string(index=False))

    best = sweep_df.loc[sweep_df["value_primary"].idxmax()]
    BEST_RANK = int(best["lora_rank"]); BEST_LR = float(best["lora_lr"])
    print(f"\nBest {short} BBBP config: rank={BEST_RANK}  lr={BEST_LR}  AUC={best['value_primary']}")

    transfer_rows = []
    for ds_name in ["BACE", "ESOL"]:
        adapter_dir = f"{ADAPTERS_DIR}/{short}_{ds_name.lower()}_r{BEST_RANK}_lr{BEST_LR:.0e}"
        print(f"\n{'#'*64}\n# {short.upper()} transfer → {ds_name} (rank={BEST_RANK} lr={BEST_LR:.0e})\n{'#'*64}")
        row = lora_train_eval(
            model_id=model_id, dataset_name=ds_name,
            rank=BEST_RANK, lr=BEST_LR, epochs=3,
            save_adapter_to=adapter_dir,
        )
        transfer_rows.append(row)
        append_lora_row(row)

    print(f"\n--- {short} transfer summary ---")
    print(pd.DataFrame(transfer_rows)[["dataset","value_primary","value_secondary","peak_mem_mb","train_time_s"]].to_string(index=False))
    return bbbp_rows + transfer_rows

## 8. SmolLM3-3B — full recipe

**Heads-up:** SmolLM3 at fp16 + gradient checkpointing should land around 9–12 GB on T4. If we OOM, the fallback is identical to 05b's plan: drop to 4-bit base (effectively QLoRA early).

In [ ]:
smolllm3_rows = run_full_recipe("HuggingFaceTB/SmolLM3-3B")

## 9. GPT-2 — full recipe

Fast: each run takes ~1–2 min. The whole 6-run recipe should finish in ~10 minutes.

In [ ]:
gpt2_rows = run_full_recipe("gpt2")

## 10. Final Phase 5 LoRA scoreboard view

In [ ]:
final = pd.read_csv(RESULTS_CSV)
print(f"LoRA scoreboard rows: {len(final)}\n")
print(final.to_string(index=False))

## 11. Done — Phase 5 fully complete

**Added in this notebook:** 12 rows — SmolLM3 (6) + GPT-2 (6).

**Total Phase 5 LoRA scoreboard: 18 rows** (6 Phi-4 from 05b + 12 here).

**What we expect to see across the three models:**
- Did LoRA beat the corresponding Phase 4 retrieval-prompting score?
- Which model benefited most from LoRA — biggest gap from prompting → fine-tuning?
- Does the best (rank, lr) match Phi-4's r=8, lr=1e-4 — or do smaller models prefer different hyperparameters?
- How does GPT-2 LoRA stack up against the 1990s Random Forest baseline (BBBP 0.681, BACE 0.860, ESOL 1.658)?

**Adapter checkpoints:** 12 new adapter folders on Google Drive at `/MyDrive/chemistry-peft-fyp/adapters/`. With 05a's pilot + 05b's 6 + 05c's 12 = **19 saved adapters** total. We'll push the best ones to HuggingFace Hub once Phase 5 closes.

**Next: Phase 6 (QLoRA).** Same recipe again, but base models loaded in 4-bit. Question: does quantising the base model hurt accuracy?